# TLT debug diagnostics

**Before:** `realize()` failed with a generic root-level message — no path, no error code.

**After:** `explain(impl)` walks the tree and reports *where* the problem is.

Run **Kernel → Restart & Run All**.

In [ ]:
import sys
sys.path.insert(0, "../src")

from pyspect import *
from pyspect.impls.str import StrImpl

impl = StrImpl(['x'])

## 1) Deep tree — invalid approximation

**Before:**
```
Exception: Invalid approximation. TLT operational semantics of formula does not hold.
```

**After:**

In [ ]:
TLT.select(ContLTL)

progress = UNTIL(AND('lane', NOT('obstacle')), 'goal')
safe_mission = AND(progress, 'battery_ok')  # intentional bug
phi = AND('mission_enabled', OR(safe_mission, 'emergency_stop'))

tree = TLT(phi, where={
    'lane': Set('LANE'), 'obstacle': Set('OBSTACLE'), 'goal': Set('GOAL'),
    'battery_ok': Set('BATTERY_OK'), 'emergency_stop': Set('EMERGENCY_STOP'),
    'mission_enabled': Set('MISSION_ON'),
})

print(tree.explain(impl))

## 2) Missing proposition binding

**Before:** `Exception: Missing proposition \`goal\``

**After:**

In [ ]:
TLT.select(ContLTL)
tree = TLT(UNTIL('safe', 'goal'), where={'safe': Set('SAFE')})
print(tree.explain(impl))

## 3) Implementation missing an operation

**Before:** `Exception: Missing from implementation: pre`

**After:**

In [ ]:
TLT.select(DiscLTL)
tree = TLT(NEXT('a'), where={'a': Set('A')})
print(tree.explain(impl))

## 4) Happy path (unchanged workflow)

In [ ]:
TLT.select(ContLTL)

phi = UNTIL(AND('lane', NOT('obstacle')), 'goal')
tree = TLT(phi, where={'lane': Set('LANE'), 'obstacle': Set('OBSTACLE'), 'goal': Set('GOAL')})

print(tree.explain(impl))
print(tree.realize(impl))